In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import linregress

In [2]:
warnings.filterwarnings("ignore")

In [3]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"

DAY4_DIR = PROJECT_DIR / "day4"
DAY4_DIR.mkdir(parents=True, exist_ok=True)

RISK_FREE_RATE = 0.065
TRADING_DAYS = 252

print("Project directory:", PROJECT_DIR)
print("Day 4 output directory:", DAY4_DIR)



Project directory: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone
Day 4 output directory: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4


In [4]:
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------

def find_csv(filename):
    """Find a CSV in processed or raw data folders."""
    locations = [
        PROCESSED_DIR / filename,
        RAW_DIR / filename,
        DATA_DIR / filename
    ]

    for path in locations:
        if path.exists():
            return path

    matches = list(PROCESSED_DIR.rglob(filename))
    if matches:
        return matches[0]

    matches = list(RAW_DIR.rglob(filename))
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"{filename} not found in data/processed or data/raw."
    )



In [5]:
def find_column(df, possible_names):
    """Find a column using possible column-name variations."""
    normalized = {
        str(col).strip().lower().replace(" ", "_"): col
        for col in df.columns
    }

    for name in possible_names:
        key = name.lower().replace(" ", "_")
        if key in normalized:
            return normalized[key]

    return None



In [6]:
def save_csv(df, filename):
    path = DAY4_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


# ------------------------------------------------------------
# 3. LOAD NAV HISTORY
# ----------

In [10]:

nav_path = find_csv("clean_nav.csv")
nav = pd.read_csv(nav_path)

print("NAV dataset loaded:", nav.shape)
print("NAV columns:", nav.columns.tolist())

# Rename columns
nav = nav.rename(columns={
    "amfi_code": "fund_name",
    "date": "date",
    "nav": "nav"
})

# Convert data types
nav["date"] = pd.to_datetime(nav["date"], errors="coerce")
nav["nav"] = pd.to_numeric(nav["nav"], errors="coerce")

# Remove missing and invalid values
nav = nav.dropna(subset=["fund_name", "date", "nav"])
nav = nav[nav["nav"] > 0]

# Sort and remove duplicates
nav = nav.sort_values(["fund_name", "date"])
nav = nav.drop_duplicates(
    subset=["fund_name", "date"],
    keep="last"
)

print("\nCleaned NAV data:")
display(nav.head())

print("Unique funds:", nav["fund_name"].nunique())


NAV dataset loaded: (46000, 3)
NAV columns: ['amfi_code', 'date', 'nav']

Cleaned NAV data:


,fund_name,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


Unique funds: 40


In [11]:
# ------------------------------------------------------------
# 4. SEPARATE BENCHMARK DATA FROM NAV HISTORY IF AVAILABLE
# ------------------------------------------------------------

# This supports benchmark rows stored in nav_history.csv.
# Otherwise, the code checks for a separate benchmark CSV.

benchmark_data = None

benchmark_mask = nav["fund_name"].astype(str).str.contains(
    r"nifty\s*50|nifty\s*100",
    case=False,
    regex=True
)

if benchmark_mask.any():
    benchmark_data = nav.loc[
        benchmark_mask, ["date", "fund_name", "nav"]
    ].copy()

    nav = nav.loc[~benchmark_mask].copy()

    print("Benchmark data found inside nav_history.csv.")

else:
    benchmark_candidates = [
        "benchmark_nav.csv",
        "nifty_nav.csv",
        "index_nav.csv"
    ]

    for filename in benchmark_candidates:
        try:
            benchmark_path = find_csv(filename)
            benchmark_data = pd.read_csv(benchmark_path)
            print("Benchmark file found:", benchmark_path)
            break
        except FileNotFoundError:
            continue



In [12]:
# 5. TASK 1 - COMPUTE DAILY RETURNS
# ------------------------------------------------------------

print("\nTASK 1: Computing daily returns...")

nav["daily_return"] = (
    nav.groupby("fund_name")["nav"].pct_change()
)

returns_computed = nav.dropna(
    subset=["daily_return"]
).copy()

# Annualised return based on daily observations
annual_returns = (
    returns_computed.groupby("fund_name")["daily_return"]
    .apply(
        lambda x: (1 + x).prod() ** (
            TRADING_DAYS / len(x)
        ) - 1
    )
    .reset_index(name="annualised_return")
)

returns_computed = returns_computed.merge(
    annual_returns,
    on="fund_name",
    how="left"
)

save_csv(returns_computed, "returns_computed.csv")

print(returns_computed.head())


# ------------


TASK 1: Computing daily returns...
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\returns_computed.csv
   fund_name       date       nav  daily_return  annualised_return
0     100016 2022-01-04  515.0971     -0.010306           0.025435
1     100016 2022-01-05  521.7239      0.012865           0.025435
2     100016 2022-01-06  515.7880     -0.011377           0.025435
3     100016 2022-01-07  515.1639     -0.001210           0.025435
4     100016 2022-01-10  510.7136     -0.008639           0.025435


In [14]:
# 6. TASK 2 - CALCULATE CAGR FOR 1, 3 AND 5 YEARS
# ------------------------------------------------------------

print("\nTASK 2: Calculating CAGR...")

cagr_results = []

for fund, group in nav.groupby("fund_name"):

    group = group.sort_values("date")

    latest_date = group["date"].max()

    for years in [1, 3, 5]:

        start_date = latest_date - pd.DateOffset(years=years)

        period_data = group[
            group["date"] >= start_date
        ]

        if (
            len(period_data) >= 2
            and period_data["date"].min() <= start_date + pd.Timedelta(days=45)
        ):

            start_nav = period_data.iloc[0]["nav"]
            end_nav = period_data.iloc[-1]["nav"]

            actual_years = (
                (period_data.iloc[-1]["date"] -
                 period_data.iloc[0]["date"]).days / 365.25
            )

            if actual_years > 0 and start_nav > 0:
                cagr = (
                    (end_nav / start_nav) ** (1 / actual_years)
                ) - 1
            else:
                cagr = np.nan

        else:
            cagr = np.nan

        cagr_results.append({
            "fund_name": fund,
            "period_years": years,
            "CAGR": cagr
        })

cagr_report = pd.DataFrame(cagr_results)

save_csv(cagr_report, "cagr_report.csv")

print(cagr_report.head(10))




TASK 2: Calculating CAGR...
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\cagr_report.csv
   fund_name  period_years      CAGR
0     100016             1 -0.022258
1     100016             3  0.012924
2     100016             5       NaN
3     100025             1  0.037076
4     100025             3  0.039155
5     100025             5       NaN
6     100033             1  0.532772
7     100033             3  0.324340
8     100033             5       NaN
9     101206             1  0.479638


In [15]:
# 7. TASK 3 - SHARPE RATIO
# ------------------------------------------------------------

print("\nTASK 3: Calculating Sharpe Ratio...")

daily_rf = RISK_FREE_RATE / TRADING_DAYS

sharpe_results = []

for fund, group in returns_computed.groupby("fund_name"):

    daily_returns = group["daily_return"].dropna()

    if len(daily_returns) > 1 and daily_returns.std() != 0:

        excess_returns = daily_returns - daily_rf

        sharpe = (
            excess_returns.mean() /
            daily_returns.std()
        ) * np.sqrt(TRADING_DAYS)

    else:
        sharpe = np.nan

    sharpe_results.append({
        "fund_name": fund,
        "sharpe_ratio": sharpe
    })

sharpe_values = pd.DataFrame(sharpe_results)

save_csv(sharpe_values, "sharpe_values.csv")

print(sharpe_values.head())


# ---------


TASK 3: Calculating Sharpe Ratio...
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\sharpe_values.csv
   fund_name  sharpe_ratio
0     100016     -0.201517
1     100025     -0.567095
2     100033      1.093699
3     101206      1.027213
4     101207      0.162661


In [16]:
#  TASK 4 - SORTINO RATIO
# ------------------------------------------------------------

print("\nTASK 4: Calculating Sortino Ratio...")

sortino_results = []

for fund, group in returns_computed.groupby("fund_name"):

    daily_returns = group["daily_return"].dropna()
    excess_returns = daily_returns - daily_rf

    downside_returns = daily_returns[
        daily_returns < daily_rf
    ] - daily_rf

    downside_deviation = np.sqrt(
        np.mean(downside_returns ** 2)
    ) if len(downside_returns) > 0 else np.nan

    if (
        pd.notna(downside_deviation)
        and downside_deviation != 0
    ):

        sortino = (
            excess_returns.mean() /
            downside_deviation
        ) * np.sqrt(TRADING_DAYS)

    else:
        sortino = np.nan

    sortino_results.append({
        "fund_name": fund,
        "sortino_ratio": sortino
    })

sortino_values = pd.DataFrame(sortino_results)

save_csv(sortino_values, "sortino_values.csv")

print(sortino_values.head())




TASK 4: Calculating Sortino Ratio...
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\sortino_values.csv
   fund_name  sortino_ratio
0     100016      -0.205599
1     100025      -0.567580
2     100033       1.121981
3     101206       1.069352
4     101207       0.165516


In [17]:
# 9. TASK 5 - ALPHA AND BETA AGAINST NIFTY 100
# ------------------------------------------------------------

print("\nTASK 5: Calculating Alpha and Beta...")

alpha_beta_results = []

nifty100_returns = None

if benchmark_data is not None:

    benchmark_date_col = find_column(
        benchmark_data, ["date", "nav_date"]
    )

    benchmark_name_col = find_column(
        benchmark_data, ["fund_name", "scheme_name", "index_name", "benchmark"]
    )

    benchmark_nav_col = find_column(
        benchmark_data, ["nav", "close", "price", "index_value"]
    )

    if (
        benchmark_date_col is not None
        and benchmark_name_col is not None
        and benchmark_nav_col is not None
    ):

        benchmark_data = benchmark_data.rename(columns={
            benchmark_date_col: "date",
            benchmark_name_col: "benchmark_name",
            benchmark_nav_col: "benchmark_nav"
        })

        benchmark_data["date"] = pd.to_datetime(
            benchmark_data["date"], errors="coerce"
        )

        benchmark_data["benchmark_nav"] = pd.to_numeric(
            benchmark_data["benchmark_nav"], errors="coerce"
        )

        benchmark_data = benchmark_data.dropna(
            subset=["date", "benchmark_nav"]
        )

        benchmark_data = benchmark_data.sort_values("date")

        nifty100 = benchmark_data[
            benchmark_data["benchmark_name"].astype(str).str.contains(
                r"nifty\s*100", case=False
            )
        ].copy()

        if not nifty100.empty:

            nifty100["benchmark_return"] = (
                nifty100["benchmark_nav"].pct_change()
            )

            nifty100_returns = nifty100[
                ["date", "benchmark_return"]
            ].dropna()

for fund, group in returns_computed.groupby("fund_name"):

    result = {
        "fund_name": fund,
        "alpha": np.nan,
        "beta": np.nan
    }

    if nifty100_returns is not None:

        fund_returns = group[["date", "daily_return"]]

        aligned = pd.merge(
            fund_returns,
            nifty100_returns,
            on="date",
            how="inner"
        ).dropna()

        if len(aligned) >= 30:

            regression = linregress(
                aligned["benchmark_return"],
                aligned["daily_return"]
            )

            result["alpha"] = regression.intercept * TRADING_DAYS
            result["beta"] = regression.slope

    alpha_beta_results.append(result)

alpha_beta = pd.DataFrame(alpha_beta_results)

save_csv(alpha_beta, "alpha_beta.csv")

print(alpha_beta.head())

if nifty100_returns is None:
    print(
        "WARNING: Nifty 100 benchmark data was not found. "
        "Alpha and Beta are saved as blank values. "
        "Add benchmark NAV data to calculate them."
    )



TASK 5: Calculating Alpha and Beta...
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\alpha_beta.csv
   fund_name  alpha  beta
0     100016    NaN   NaN
1     100025    NaN   NaN
2     100033    NaN   NaN
3     101206    NaN   NaN
4     101207    NaN   NaN


In [18]:
# 10. TASK 6 - MAXIMUM DRAWDOWN
# ------------------------------------------------------------

print("\nTASK 6: Calculating Maximum Drawdown...")

drawdown_results = []

for fund, group in nav.groupby("fund_name"):

    group = group.sort_values("date").copy()

    group["running_max"] = group["nav"].cummax()

    group["drawdown"] = (
        group["nav"] / group["running_max"]
    ) - 1

    worst_index = group["drawdown"].idxmin()
    worst_row = group.loc[worst_index]

    peak_data = group[
        (group["date"] <= worst_row["date"]) &
        (group["nav"] == worst_row["running_max"])
    ]

    peak_date = (
        peak_data.iloc[-1]["date"]
        if not peak_data.empty else pd.NaT
    )

    drawdown_results.append({
        "fund_name": fund,
        "maximum_drawdown": worst_row["drawdown"],
        "peak_date": peak_date,
        "trough_date": worst_row["date"]
    })

max_drawdown = pd.DataFrame(drawdown_results)

save_csv(max_drawdown, "max_drawdown.csv")

print(max_drawdown.head())




TASK 6: Calculating Maximum Drawdown...
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\max_drawdown.csv
   fund_name  maximum_drawdown  peak_date trough_date
0     100016         -0.247344 2022-03-30  2022-09-15
1     100025         -0.043083 2023-05-23  2023-07-28
2     100033         -0.162172 2022-03-11  2022-05-12
3     101206         -0.112916 2023-04-24  2023-07-05
4     101207         -0.354469 2024-11-21  2026-05-11


In [19]:
# 11. TASK 7 - FUND SCORECARD (0-100)
# ------------------------------------------------------------

print("\nTASK 7: Building Fund Scorecard...")

# Get 3-year CAGR
cagr_3yr = cagr_report[
    cagr_report["period_years"] == 3
][["fund_name", "CAGR"]].rename(
    columns={"CAGR": "three_year_return"}
)

scorecard = cagr_3yr.merge(
    sharpe_values, on="fund_name", how="outer"
)

scorecard = scorecard.merge(
    alpha_beta[["fund_name", "alpha"]],
    on="fund_name",
    how="left"
)

scorecard = scorecard.merge(
    max_drawdown[["fund_name", "maximum_drawdown"]],
    on="fund_name",
    how="left"
)

# Load fund master for expense ratio and category
try:
    fund_master = pd.read_csv(find_csv("fund_master.csv"))

    master_fund_col = find_column(
        fund_master, ["scheme_name", "fund_name", "scheme"]
    )

    expense_col = find_column(
        fund_master, ["expense_ratio", "expense_ratio_percent", "ter"]
    )

    category_col = find_column(
        fund_master, ["category", "scheme_category", "fund_category"]
    )

    if master_fund_col is not None:

        fund_master = fund_master.rename(
            columns={master_fund_col: "fund_name"}
        )

        selected_cols = ["fund_name"]

        if expense_col is not None:
            fund_master = fund_master.rename(
                columns={expense_col: "expense_ratio"}
            )
            selected_cols.append("expense_ratio")

        if category_col is not None:
            fund_master = fund_master.rename(
                columns={category_col: "category"}
            )
            selected_cols.append("category")

        scorecard = scorecard.merge(
            fund_master[selected_cols].drop_duplicates("fund_name"),
            on="fund_name",
            how="left"
        )

except FileNotFoundError:
    print("WARNING: fund_master.csv not found.")

if "expense_ratio" not in scorecard.columns:
    scorecard["expense_ratio"] = np.nan

if "category" not in scorecard.columns:
    scorecard["category"] = "Not Available"


# Percentile ranking: higher score = better performance
def percentile_score(series, higher_is_better=True):

    values = pd.to_numeric(series, errors="coerce")

    if values.notna().sum() < 2:
        return pd.Series(np.nan, index=series.index)

    return (
        values.rank(
            pct=True,
            ascending=higher_is_better
        ) * 100
    )


scorecard["return_score"] = percentile_score(
    scorecard["three_year_return"], True
)

scorecard["sharpe_score"] = percentile_score(
    scorecard["sharpe_ratio"], True
)

scorecard["alpha_score"] = percentile_score(
    scorecard["alpha"], True
)

scorecard["expense_score"] = percentile_score(
    scorecard["expense_ratio"], False
)

scorecard["drawdown_score"] = percentile_score(
    scorecard["maximum_drawdown"], True
)

# Apply the weights from the project instructions.
# Missing metrics are excluded and remaining weights are rescaled.

score_columns = {
    "return_score": 0.30,
    "sharpe_score": 0.25,
    "alpha_score": 0.20,
    "expense_score": 0.15,
    "drawdown_score": 0.10
}

weighted_values = pd.DataFrame(index=scorecard.index)
weighted_weights = pd.DataFrame(index=scorecard.index)

for col, weight in score_columns.items():

    weighted_values[col] = scorecard[col].fillna(0) * weight
    weighted_weights[col] = scorecard[col].notna() * weight

scorecard["composite_score"] = (
    weighted_values.sum(axis=1) /
    weighted_weights.sum(axis=1).replace(0, np.nan)
)

scorecard["composite_score"] = scorecard[
    "composite_score"
].round(2)

scorecard = scorecard.sort_values(
    "composite_score", ascending=False
)

scorecard["rank"] = range(1, len(scorecard) + 1)

save_csv(scorecard, "fund_scorecard.csv")

print("\nTop 10 funds:")
print(
    scorecard[
        ["rank", "fund_name", "three_year_return",
         "sharpe_ratio", "alpha", "composite_score"]
    ].head(10)
)


# -----


TASK 7: Building Fund Scorecard...
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\fund_scorecard.csv

Top 10 funds:
    rank  fund_name  three_year_return  sharpe_ratio  alpha  composite_score
34     1     148567           0.339920      1.448291    NaN            96.15
30     2     120843           0.295751      1.306744    NaN            87.50
19     3     119551           0.304486      1.208267    NaN            85.19
24     4     120504           0.324789      1.026524    NaN            84.81
2      5     100033           0.324340      1.093699    NaN            83.46
25     6     120505           0.317692      1.180101    NaN            82.31
36     7     148569           0.291714      1.234930    NaN            82.31
3      8     101206           0.289602      1.027213    NaN            80.00
16     9     119094           0.351025      0.998231    NaN            80.00
38    10     149323           0.268631      1.132122    NaN            75.58


In [21]:
# 12. TASK 8 - BENCHMARK COMPARISON AND TRACKING ERROR
# ------------------------------------------------------------

print("\nTASK 8: Benchmark comparison...")

tracking_error_results = []

if benchmark_data is not None and nifty100_returns is not None:

    nifty50 = benchmark_data[
        benchmark_data["benchmark_name"].astype(str).str.contains(
            r"nifty\s*50", case=False
        )
    ].copy()

    nifty100 = benchmark_data[
        benchmark_data["benchmark_name"].astype(str).str.contains(
            r"nifty\s*100", case=False
        )
    ].copy()

    if not nifty50.empty and not nifty100.empty:

        nifty50 = nifty50.sort_values("date")
        nifty100 = nifty100.sort_values("date")

        latest_date = min(
            nav["date"].max(),
            nifty50["date"].max(),
            nifty100["date"].max()
        )

        start_date = latest_date - pd.DateOffset(years=3)

        nifty50 = nifty50[
            (nifty50["date"] >= start_date) &
            (nifty50["date"] <= latest_date)
        ]

        nifty100 = nifty100[
            (nifty100["date"] >= start_date) &
            (nifty100["date"] <= latest_date)
        ]

        top_funds = scorecard[
            "fund_name"
        ].head(5).tolist()

        plt.figure(figsize=(14, 7))

        for fund in top_funds:

            fund_data = nav[
                (nav["fund_name"] == fund) &
                (nav["date"] >= start_date) &
                (nav["date"] <= latest_date)
            ].sort_values("date").copy()

            if not fund_data.empty:

                fund_data["indexed_value"] = (
                    fund_data["nav"] /
                    fund_data["nav"].iloc[0]
                ) * 100

                plt.plot(
                    fund_data["date"],
                    fund_data["indexed_value"],
                    label=fund
                )

                # Tracking error against Nifty 100
                benchmark_returns = nifty100[
                    ["date", "benchmark_nav"]
                ].copy()

                benchmark_returns["benchmark_return"] = (
                    benchmark_returns["benchmark_nav"].pct_change()
                )

                aligned = pd.merge(
                    fund_data[["date", "nav"]],
                    benchmark_returns[["date", "benchmark_return"]],
                    on="date",
                    how="inner"
                )

                aligned["fund_return"] = aligned["nav"].pct_change()

                active_returns = (
                    aligned["fund_return"] -
                    aligned["benchmark_return"]
                ).dropna()

                tracking_error = (
                    active_returns.std() * np.sqrt(TRADING_DAYS)
                    if len(active_returns) > 1
                    else np.nan
                )

                tracking_error_results.append({
                    "fund_name": fund,
                    "benchmark": "Nifty 100",
                    "tracking_error": tracking_error
                })

        # Benchmark indexed values
        for benchmark, label in [
            (nifty50, "Nifty 50"),
            (nifty100, "Nifty 100")
        ]:

            benchmark = benchmark.sort_values("date").copy()

            benchmark["indexed_value"] = (
                benchmark["benchmark_nav"] /
                benchmark["benchmark_nav"].iloc[0]
            ) * 100

            plt.plot(
                benchmark["date"],
                benchmark["indexed_value"],
                linestyle="--",
                linewidth=2,
                label=label
            )

        plt.title(
            "Top 5 Mutual Funds vs Nifty 50 and Nifty 100 (3 Years)"
        )
        plt.xlabel("Date")
        plt.ylabel("Growth of ₹100")
        plt.legend(loc="best")
        plt.grid(alpha=0.3)
        plt.tight_layout()

        chart_path = DAY4_DIR / "benchmark_chart.png"
        plt.savefig(chart_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("Benchmark chart saved:", chart_path)

    else:
        print(
            "Nifty 50 or Nifty 100 data is missing. "
            "Benchmark chart could not be generated."
        )

else:
    print(
        "Benchmark data is unavailable. "
        "Add Nifty 50 and Nifty 100 NAV data to generate "
        "the benchmark chart and tracking errors."
    )

tracking_error_df = pd.DataFrame(tracking_error_results)

save_csv(
    tracking_error_df,
    "tracking_error.csv"
)




TASK 8: Benchmark comparison...
Benchmark data is unavailable. Add Nifty 50 and Nifty 100 NAV data to generate the benchmark chart and tracking errors.
Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\tracking_error.csv


In [22]:
# 13. SAVE ADDITIONAL ANALYSIS SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame({
    "Metric": [
        "Total Funds Analysed",
        "NAV Records",
        "Start Date",
        "End Date",
        "Risk Free Rate",
        "Trading Days Per Year"
    ],
    "Value": [
        nav["fund_name"].nunique(),
        len(nav),
        nav["date"].min(),
        nav["date"].max(),
        RISK_FREE_RATE,
        TRADING_DAYS
    ]
})

save_csv(summary, "performance_summary.csv")


# ------------------------------------------------------------
# 14. FINAL OUTPUT CHECK
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DAY 4 PERFORMANCE ANALYTICS COMPLETED")
print("=" * 60)

print("\nFiles generated inside:", DAY4_DIR)

for file in sorted(DAY4_DIR.iterdir()):
    print(file.name)

print("\nTop 5 funds by composite score:")
print(
    scorecard[
        ["rank", "fund_name", "composite_score"]
    ].head(5).to_string(index=False)
)


Saved: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4\performance_summary.csv

DAY 4 PERFORMANCE ANALYTICS COMPLETED

Files generated inside: d:\DataAnalytics At BlueStock\capstone project-1\bluestock_mf_capstone\day4
alpha_beta.csv
cagr_report.csv
fund_scorecard.csv
max_drawdown.csv
performance_summary.csv
returns_computed.csv
sharpe_values.csv
sortino_values.csv
tracking_error.csv

Top 5 funds by composite score:
 rank  fund_name  composite_score
    1     148567            96.15
    2     120843            87.50
    3     119551            85.19
    4     120504            84.81
    5     100033            83.46
